# 🎵 소스 분리 — 음원에서 악기별 트랙 분리

여러 악기가 섞인 음원에서 **피아노만 추출**하거나, **반주만 남길** 수 있습니다.
합주 연습용 반주 트랙을 만들거나, 좋아하는 곡의 피아노 파트만 따로 들어볼 수 있습니다.

**도구**: [Demucs](https://github.com/facebookresearch/demucs) — Meta가 만든 오픈소스 소스 분리 AI

⏰ 시간이 부족하면 이 노트북은 수업 후에 해보세요. Colab에 다 준비되어 있습니다.

▶ 먼저 아래 셀을 실행해주세요. 설치에 2~3분 정도 걸립니다.

In [ ]:
%%capture
!pip install -q demucs librosa soundfile matplotlib

import warnings
warnings.filterwarnings('ignore')

print('✅ 설치 완료!')

---
## 1. 음원 올리기

▶ 여러 악기가 섞인 음원을 업로드하세요.

피아노 협주곡, 밴드 음악, 보컬이 있는 곡 등 악기가 여러 개 섞인 음원이 좋습니다.

음원이 없으면 이 셀은 건너뛰고, 다음 셀에서 예시 파일을 사용하세요.

In [ ]:
from google.colab import files
import os

uploaded = files.upload()
input_audio = list(uploaded.keys())[0]
print(f'✅ 파일이 업로드되었습니다: {input_audio}')

---
## 2. (선택) 예시 파일 사용

▶ 음원이 없으면 이 셀을 실행하세요. 피아노 협주곡 예시를 다운로드합니다.

In [ ]:
import urllib.request
import IPython.display as ipd

REPO_URL = 'https://raw.githubusercontent.com/joonhyungbae/pianokit/main/assets'
input_audio = 'concerto_example.wav'

if not os.path.exists(input_audio):
    try:
        urllib.request.urlretrieve(f'{REPO_URL}/{input_audio}', input_audio)
        print(f'✅ 다운로드 완료: {input_audio}')
    except:
        print(f'⚠️ 다운로드 실패. 위 셀에서 직접 업로드해주세요.')

print(f'\n🎵 원본 음원:')
if os.path.exists(input_audio):
    ipd.display(ipd.Audio(input_audio))

---
## 3. AI로 악기별 분리하기

Demucs는 음원의 파형 패턴을 분석해서, 각 악기가 어떤 소리를 내고 있는지 구분합니다.
하나의 음원에서 **보컬, 드럼, 베이스, 기타/피아노** 네 가지 트랙을 분리합니다.

⏰ 1~2분 정도 걸립니다. ▶ 아래 셀을 실행하세요.

In [ ]:
import subprocess

print('🔄 AI가 악기를 분리하고 있습니다... (1~2분 소요)')
print('   htdemucs_ft (fine-tuned) 모델 사용 — Demucs v4 최고 품질')

# htdemucs_ft: fine-tuned 모델 (4-stem: vocals, drums, bass, other)
result = subprocess.run(
    ['python', '-m', 'demucs', '-n', 'htdemucs_ft', input_audio],
    capture_output=True, text=True
)

# Demucs는 separated/htdemucs_ft/{파일명}/ 디렉토리에 결과 저장
stem_name = os.path.splitext(os.path.basename(input_audio))[0]
output_dir = f'separated/htdemucs_ft/{stem_name}'

if os.path.exists(output_dir):
    tracks = os.listdir(output_dir)
    print(f'✅ 분리 완료! {len(tracks)}개 트랙:')
    for t in sorted(tracks):
        print(f'  🎵 {t}')
else:
    print('⚠️ 분리에 실패했습니다.')
    print(result.stderr[-500:] if result.stderr else '에러 정보 없음')

---
## 4. 분리된 트랙 들어보기

아래에서 각 트랙을 따로 들어볼 수 있습니다.
원본과 번갈아 들으면서 얼마나 깔끔하게 분리되었는지 확인해보세요.

In [ ]:
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import numpy as np
import IPython.display as ipd

track_labels = {
    'vocals.wav': '🎤 보컬',
    'drums.wav': '🥁 드럼',
    'bass.wav': '🎸 베이스',
    'other.wav': '🎹 기타/피아노',
    'no_vocals.wav': '🎶 보컬 제거 (반주)',
}

if os.path.exists(output_dir):
    tracks = sorted(os.listdir(output_dir))
    fig, axes = plt.subplots(len(tracks), 1, figsize=(14, 3 * len(tracks)))
    if len(tracks) == 1:
        axes = [axes]

    for idx, track_file in enumerate(tracks):
        track_path = os.path.join(output_dir, track_file)
        label = track_labels.get(track_file, track_file)

        # 파형 시각화
        y, sr = librosa.load(track_path, sr=None, mono=True)
        times = np.arange(len(y)) / sr
        axes[idx].plot(times, y, linewidth=0.3, color='#4FC3F7')
        axes[idx].set_title(label, fontsize=12)
        axes[idx].set_xlim(0, times[-1])
        axes[idx].set_ylabel('진폭')

        # 재생 위젯
        print(f'\n{label}:')
        ipd.display(ipd.Audio(track_path))

    axes[-1].set_xlabel('시간 (초)')
    plt.tight_layout()
    plt.show()
else:
    print('⚠️ 분리된 트랙을 찾을 수 없습니다. 3번 셀을 먼저 실행해주세요.')

---
## 5. 트랙 다운로드

▶ 원하는 트랙을 다운로드하세요.

분리한 트랙은 '텍스트 → 음악'이나 'AI 오디오리액티브' 노트북의 입력으로도 활용할 수 있습니다.

In [ ]:
import zipfile
from google.colab import files

if os.path.exists(output_dir):
    # 전체 zip으로 묶기
    zip_name = f'{stem_name}_separated.zip'
    with zipfile.ZipFile(zip_name, 'w') as zf:
        for track_file in os.listdir(output_dir):
            track_path = os.path.join(output_dir, track_file)
            zf.write(track_path, track_file)

    print(f'📦 전체 트랙 다운로드:')
    files.download(zip_name)
else:
    print('⚠️ 분리된 트랙이 없습니다.')